In [ ]:
import os
from pyspark.sql import SparkSession

# Список зависимостей Spark
spark_packages = [
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2",
        "org.postgresql:postgresql:42.5.0",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0"
]

# Создание SparkSession
spark_builder = SparkSession.builder \
        .appName("SparkExample") \
        .config("spark.jars.packages", ",".join(spark_packages))
    
# Инициализация сессии
spark = spark_builder.getOrCreate()



In [ ]:
# ⬇️ Параметры подключения к PostgreSQL
from pyspark.sql import functions as F
from pyspark.sql.window import Window

jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shops_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shops") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

shop_tz_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shop_timezone") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

# Регистрируем DataFrame-ы как временные вьюхи
#shops_df.createOrReplaceTempView("shops")
#shop_tz_df.createOrReplaceTempView("shop_timezone")
#shop_tz_df.show()
#shops_df.show()

# Преобрахование через PySpark
# 1. CTE 'sz' - преобразование shop_timezone
# SELECT cast(plant as INT) as id, time_zone FROM shop_timezone where cast(plant as INT) IS NOT NULL
sz_df = shop_tz_df.select(
        F.col("plant").cast("int").alias("id"),
        F.col("time_zone")
    ).filter(F.col("plant").cast("int").isNotNull())
    
# 2. CTE 's' - преобразование shops
# select cast(st_id AS INT) as st_id, shop_name from shops
s_df = shops_df.select(
        F.col("st_id").cast("int").alias("st_id"),
        F.col("shop_name")
    )
    
# 3. CTE 'cte' - объединение с ранжированием
# select *, row_number() over(PARTITION BY st_id ORDER BY st_id) as rnk from s join sz ON s.st_id = sz.id
joined_df = s_df.join(sz_df, s_df["st_id"] == sz_df["id"], "inner")
    
# Оконная спецификация - ORDER BY st_id в оригинальном SQL
window_spec = Window.partitionBy("st_id").orderBy("st_id")
cte_df = joined_df.withColumn("rnk", F.row_number().over(window_spec))
    
# 4. Финальный SELECT
# select st_id, shop_name, CAST(CASE WHEN time_zone = '' OR time_zone IS NULL THEN 3 
# ELSE substr(time_zone, 4) END AS INT) AS tz_code from cte where rnk = 1

    
result_df = cte_df.filter(F.col("rnk") == 1).select(
        F.col("st_id"),
        F.col("shop_name"),
        F.when(
            (F.col("time_zone") == "") | F.col("time_zone").isNull(),
            F.lit(3)
        ).otherwise(
            F.expr("substring(time_zone, 4)").cast("int")
        ).alias("tz_code")
    )

result_df.show()



In [10]:
spark.stop()